# Model 5 — Swin Transformer + Segmentation
**Plant Leaf Disease Classification Dataset**  
Dataset: https://www.kaggle.com/datasets/mertcangelbal/plant-leaf-disease-classification-dataset  
GPU: RTX 5090 (Vast.ai)

**Yaklaşım:** Swin-B backbone üzerine segmentation head eklenir.  
Swin'in hierarchical feature map'leri (4 stage) segmentation için idealdir.  
Son stage feature map'i üzerinden saliency map üretilir.

In [ ]:
# ── Dataset İndirme ──────────────────────────────────────────────────────────
!pip install -q kagglehub timm

import kagglehub, shutil, os
from pathlib import Path

_raw = kagglehub.dataset_download("mertcangelbal/plant-leaf-disease-classification-dataset")
print("Kaggle cache yolu:", _raw)

DATA_ROOT = Path("../data")
if not DATA_ROOT.exists():
    print(f"Dataset {DATA_ROOT} dizinine kopyalanıyor...")
    shutil.copytree(_raw, str(DATA_ROOT))
    print("Kopyalama tamamlandı.")
else:
    print(f"{DATA_ROOT} zaten mevcut, indirme atlandı.")

In [ ]:
# ── Kütüphaneler ─────────────────────────────────────────────────────────────
import os, time, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# ── Konfigürasyon ─────────────────────────────────────────────────────────────
CONFIG = {
    "data_dir":        "../data",
    "img_size":        224,
    "batch_size":      32,
    "num_classes":     105,
    "epochs":          50,
    "lr":              3e-4,
    "weight_decay":    1e-4,
    "num_workers":     4,
    "seg_loss_weight": 0.3,
    "save_path":       "./checkpoints/swin_seg_best.pth",
}
os.makedirs("./checkpoints", exist_ok=True)
os.makedirs("./results", exist_ok=True)

In [ ]:
# ── Veri Ön İşleme ────────────────────────────────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomCrop(CONFIG["img_size"], padding=8),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transforms = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

data_dir = Path(CONFIG["data_dir"])
train_dataset = datasets.ImageFolder(data_dir / "train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(data_dir / "val",   transform=val_transforms)
test_dataset  = datasets.ImageFolder(data_dir / "test",  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                          shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"],
                          shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Train: {len(train_dataset):,} | Val: {len(val_dataset):,} | Test: {len(test_dataset):,}")

In [ ]:
# ── Class Weights ─────────────────────────────────────────────────────────────
class_counts = np.array([len(list((data_dir / "train" / c).glob("*")))
                         for c in train_dataset.classes], dtype=np.float32)
class_counts = np.maximum(class_counts, 1)
class_weights = torch.tensor(
    len(train_dataset) / (CONFIG["num_classes"] * class_counts),
    dtype=torch.float32
).to(DEVICE)
print(f"class_weights shape: {class_weights.shape}")  # (105,) olmalı

In [ ]:
# ── Swin + Segmentation Mimarisi ──────────────────────────────────────────────
class SwinWithSegmentation(nn.Module):
    """
    Swin-B backbone + segmentation head.
    Swin'in son stage feature map'i (7x7) üzerinden saliency map üretilir.
    """
    def __init__(self, num_classes=105, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=pretrained,
            num_classes=0,
            global_pool="",
        )
        # Swin-B son stage çıkışı: [B, 7*7, 1024]
        feat_dim = 1024

        # Segmentation head
        self.seg_head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Linear(256, 1),
        )

        # Sınıflandırma head
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, num_classes),
        )

    def forward(self, x):
        feat = self.backbone.forward_features(x)  # [B, 49, 1024] veya [B, 1024]

        # timm sürümüne göre çıktı 2D veya 3D olabilir
        if feat.dim() == 2:
            # [B, 1024] — global pool yapılmış
            fused = feat
            seg_scores = torch.zeros(feat.size(0), 49, device=feat.device)
        else:
            # [B, 49, 1024]
            seg_scores = self.seg_head(feat).squeeze(-1)        # [B, 49]
            seg_attn   = torch.softmax(seg_scores, dim=-1)       # [B, 49]
            weighted   = (seg_attn.unsqueeze(-1) * feat).sum(dim=1)  # [B, 1024]
            global_feat = feat.mean(dim=1)                       # [B, 1024]
            fused = global_feat + weighted

        logits = self.classifier(fused)
        return logits, seg_scores

model = SwinWithSegmentation(num_classes=CONFIG["num_classes"]).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre: {total_params:,}")

In [ ]:
# ── Loss & Optimizer ─────────────────────────────────────────────────────────
cls_criterion = nn.CrossEntropyLoss(weight=class_weights)

def seg_entropy_loss(seg_scores):
    probs = torch.softmax(seg_scores, dim=-1)
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=-1).mean()
    return -entropy

optimizer = optim.AdamW(model.parameters(),
                        lr=CONFIG["lr"],
                        weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                  T_max=CONFIG["epochs"],
                                                  eta_min=1e-6)

In [ ]:
# ── Eğitim Döngüsü ────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, seg_scores = model(imgs)
        loss_cls = cls_criterion(logits, labels)
        loss_seg = seg_entropy_loss(seg_scores)
        loss = loss_cls + CONFIG["seg_loss_weight"] * loss_seg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss_cls.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits, _ = model(imgs)
        loss = cls_criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
train_start = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, DEVICE)
    vl_loss, vl_acc = evaluate(model, val_loader, DEVICE)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_loss"].append(vl_loss)
    history["val_acc"].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), CONFIG["save_path"])

    print(f"Epoch [{epoch:02d}/{CONFIG['epochs']}] "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f} | "
          f"{time.time()-t0:.1f}s")

total_train_time = time.time() - train_start
print(f"\nToplam eğitim süresi: {total_train_time/60:.1f} dakika")

In [ ]:
# ── Segmentation Map Görselleştirme ───────────────────────────────────────────
model.load_state_dict(torch.load(CONFIG["save_path"]))
model.eval()

sample_imgs, sample_labels = next(iter(test_loader))
with torch.no_grad():
    _, seg_scores = model(sample_imgs[:4].to(DEVICE))

seg_maps = torch.softmax(seg_scores[:4], dim=-1).cpu()
n_patches = int(seg_maps.shape[1] ** 0.5)  # 7

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img = sample_imgs[i].permute(1, 2, 0).numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    axes[0, i].imshow(img); axes[0, i].axis("off")
    axes[0, i].set_title(train_dataset.classes[sample_labels[i]][:15])
    smap = seg_maps[i].reshape(n_patches, n_patches).numpy()
    axes[1, i].imshow(smap, cmap="hot"); axes[1, i].axis("off")
    axes[1, i].set_title("Seg Map")

plt.suptitle("Swin+Seg — Segmentation Maps")
plt.tight_layout()
plt.savefig("./results/swin_seg_maps.png", dpi=150)
plt.show()

In [ ]:
# ── Test Değerlendirmesi ──────────────────────────────────────────────────────
all_preds, all_labels = [], []
infer_start = time.time()

with torch.no_grad():
    for imgs, labels in test_loader:
        logits, _ = model(imgs.to(DEVICE))
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

infer_time = time.time() - infer_start
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
rec  = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
f1   = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

print(f"Test Accuracy:  {acc:.4f}")
print(f"Precision:      {prec:.4f}")
print(f"Recall:         {rec:.4f}")
print(f"F1 Score:       {f1:.4f}")
print(f"Inference Time: {infer_time/len(test_dataset)*1000:.2f} ms/image")

MODEL_LABEL  = "Swin Transformer + Segmentation"
MODEL_PREFIX = "swin_seg"
class_names  = train_dataset.classes

# ── 1. Per-Class F1 Score ─────────────────────────────────────────────────────
report_dict = classification_report(all_labels, all_preds,
                                    target_names=class_names,
                                    output_dict=True, zero_division=0)
per_class_f1 = {k: v["f1-score"] for k, v in report_dict.items()
                if k not in ("accuracy", "macro avg", "weighted avg")}
sorted_f1 = sorted(per_class_f1.items(), key=lambda x: x[1])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
worst20 = sorted_f1[:20]
best20  = sorted_f1[-20:][::-1]
axes[0].barh([x[0][:25] for x in worst20], [x[1] for x in worst20], color="#EF5350")
axes[0].set_title(f"{MODEL_LABEL} — En Düşük F1 (20 Sınıf)")
axes[0].set_xlabel("F1 Score"); axes[0].set_xlim(0, 1)
axes[1].barh([x[0][:25] for x in best20],  [x[1] for x in best20],  color="#66BB6A")
axes[1].set_title(f"{MODEL_LABEL} — En Yüksek F1 (20 Sınıf)")
axes[1].set_xlabel("F1 Score"); axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_per_class_f1.png", dpi=150)
plt.show()

# ── 2. Top-10 Karıştırılan Sınıf Çiftleri ────────────────────────────────────
cm_full = confusion_matrix(all_labels, all_preds)
np.fill_diagonal(cm_full, 0)
flat_idx = np.argsort(cm_full.ravel())[::-1][:10]
rows_e, cols_e = np.unravel_index(flat_idx, cm_full.shape)

fig, ax = plt.subplots(figsize=(12, 5))
labels_top = [f"{class_names[r][:15]}→{class_names[c][:15]}" for r, c in zip(rows_e, cols_e)]
counts_top = [cm_full[r, c] for r, c in zip(rows_e, cols_e)]
bars = ax.barh(labels_top[::-1], counts_top[::-1], color="#5C6BC0")
for bar, val in zip(bars, counts_top[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
ax.set_title(f"{MODEL_LABEL} — En Çok Karıştırılan 10 Sınıf Çifti")
ax.set_xlabel("Yanlış Sınıflandırma Sayısı")
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_top_errors.png", dpi=150)
plt.show()

# ── 3. Makro-Ortalama ROC Eğrisi ─────────────────────────────────────────────
all_probs_ss = []
model.eval()
with torch.no_grad():
    for imgs, _ in test_loader:
        logits_r, _ = model(imgs.to(DEVICE))
        all_probs_ss.extend(torch.softmax(logits_r, dim=1).cpu().numpy())
all_probs_ss = np.array(all_probs_ss)

labels_bin = label_binarize(all_labels, classes=list(range(CONFIG["num_classes"])))
fpr_dict, tpr_dict, roc_auc_dict = {}, {}, {}
for i in range(CONFIG["num_classes"]):
    if labels_bin[:, i].sum() > 0:
        fpr_dict[i], tpr_dict[i], _ = roc_curve(labels_bin[:, i], all_probs_ss[:, i])
        roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

all_fpr  = np.unique(np.concatenate([fpr_dict[i] for i in roc_auc_dict]))
mean_tpr = np.zeros_like(all_fpr)
for i in roc_auc_dict:
    mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
mean_tpr /= len(roc_auc_dict)
macro_auc = auc(all_fpr, mean_tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(all_fpr, mean_tpr, color="#1565C0", lw=2,
        label=f"Makro-Ortalama ROC (AUC = {macro_auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"{MODEL_LABEL} — ROC Eğrisi (Makro Ortalama)")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_roc.png", dpi=150)
plt.show()

print(f"\nMakro-Ortalama ROC AUC: {macro_auc:.4f}")
print("Grafikler ./results/ dizinine kaydedildi.")

In [ ]:
# ── Kapsamlı Grafiksel Değerlendirme ─────────────────────────────────────────
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import warnings; warnings.filterwarnings("ignore")

MODEL_LABEL  = "Swin Transformer + Segmentation"
MODEL_PREFIX = "swin_seg"
class_names  = train_dataset.classes

# ── 1. Per-Class F1 Score ─────────────────────────────────────────────────────
report_dict = classification_report(all_labels, all_preds,
                                    target_names=class_names,
                                    output_dict=True, zero_division=0)
per_class_f1 = {k: v["f1-score"] for k, v in report_dict.items()
                if k not in ("accuracy", "macro avg", "weighted avg")}
sorted_f1 = sorted(per_class_f1.items(), key=lambda x: x[1])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
worst20 = sorted_f1[:20]
best20  = sorted_f1[-20:][::-1]
axes[0].barh([x[0][:25] for x in worst20], [x[1] for x in worst20], color="#EF5350")
axes[0].set_title(f"{MODEL_LABEL} — En Düşük F1 (20 Sınıf)")
axes[0].set_xlabel("F1 Score"); axes[0].set_xlim(0, 1)
axes[1].barh([x[0][:25] for x in best20],  [x[1] for x in best20],  color="#66BB6A")
axes[1].set_title(f"{MODEL_LABEL} — En Yüksek F1 (20 Sınıf)")
axes[1].set_xlabel("F1 Score"); axes[1].set_xlim(0, 1)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_per_class_f1.png", dpi=150)
plt.show()

# ── 2. Top-10 Karıştırılan Sınıf Çiftleri ────────────────────────────────────
cm_full = confusion_matrix(all_labels, all_preds)
np.fill_diagonal(cm_full, 0)
flat_idx = np.argsort(cm_full.ravel())[::-1][:10]
rows_e, cols_e = np.unravel_index(flat_idx, cm_full.shape)

fig, ax = plt.subplots(figsize=(12, 5))
labels_top = [f"{class_names[r][:15]}→{class_names[c][:15]}" for r, c in zip(rows_e, cols_e)]
counts_top = [cm_full[r, c] for r, c in zip(rows_e, cols_e)]
bars = ax.barh(labels_top[::-1], counts_top[::-1], color="#5C6BC0")
for bar, val in zip(bars, counts_top[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
ax.set_title(f"{MODEL_LABEL} — En Çok Karıştırılan 10 Sınıf Çifti")
ax.set_xlabel("Yanlış Sınıflandırma Sayısı")
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_top_errors.png", dpi=150)
plt.show()

# ── 3. Makro-Ortalama ROC Eğrisi ─────────────────────────────────────────────
all_probs_ss = []
model.eval()
with torch.no_grad():
    for imgs, _ in test_loader:
        logits_r, _ = model(imgs.to(DEVICE))
        all_probs_ss.extend(torch.softmax(logits_r, dim=1).cpu().numpy())
all_probs_ss = np.array(all_probs_ss)

labels_bin = label_binarize(all_labels, classes=list(range(CONFIG["num_classes"])))
fpr_dict, tpr_dict, roc_auc_dict = {}, {}, {}
for i in range(CONFIG["num_classes"]):
    if labels_bin[:, i].sum() > 0:
        fpr_dict[i], tpr_dict[i], _ = roc_curve(labels_bin[:, i], all_probs_ss[:, i])
        roc_auc_dict[i] = auc(fpr_dict[i], tpr_dict[i])

all_fpr  = np.unique(np.concatenate([fpr_dict[i] for i in roc_auc_dict]))
mean_tpr = np.zeros_like(all_fpr)
for i in roc_auc_dict:
    mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
mean_tpr /= len(roc_auc_dict)
macro_auc = auc(all_fpr, mean_tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(all_fpr, mean_tpr, color="#1565C0", lw=2,
        label=f"Makro-Ortalama ROC (AUC = {macro_auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"{MODEL_LABEL} — ROC Eğrisi (Makro Ortalama)")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"./results/{MODEL_PREFIX}_roc.png", dpi=150)
plt.show()

print(f"\nMakro-Ortalama ROC AUC: {macro_auc:.4f}")
print("Grafikler ./results/ dizinine kaydedildi.")

In [ ]:
# ── Sonuçları Kaydet ──────────────────────────────────────────────────────────
results = {
    "model": "Swin Transformer + Segmentation",
    "accuracy":       round(acc, 4),
    "precision":      round(prec, 4),
    "recall":         round(rec, 4),
    "f1_score":       round(f1, 4),
    "roc_auc_macro":  round(macro_auc, 4),
    "training_time_min": round(total_train_time / 60, 2),
    "inference_time_ms_per_image": round(infer_time / len(test_dataset) * 1000, 4),
    "best_val_acc":   round(best_val_acc, 4),
    "total_params":   total_params,
    "config":         CONFIG,
}
with open("./results/swin_seg_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

## Mobil Uyumluluk Notu — Swin Transformer + Segmentation

| Özellik | Değer |
|---------|-------|
| Mimari | Swin-B + Segmentation Head |
| Parametre sayısı | ~88M + ~500K (seg head) |
| Model boyutu (.pth) | ~345 MB |
| Inference süresi (GPU) | ~14–22 ms/görüntü |
| Mobil uygunluğu | ⚠️ Orta |

**Avantajlar:**
- Hierarchical feature map'ler sayesinde segmentation head daha anlamlı spatial attention üretir
- ViT+Seg'e kıyasla daha verimli attention hesabı
- 7×7 spatial map → görselleştirme için yeterli çözünürlük

**Potansiyel kısıtlar:**
- Temel Swin-B ile aynı ağırlık maliyeti
- Mobil için Swin-T + Seg kombinasyonu öncelikli değerlendirilmeli

**Mobil dönüşüm için önerilen araçlar:** ONNX export + INT8 quantization, spatial attention map ayrı endpoint olarak sunulabilir